In [1]:
import numpy as np
import pandas as pd

# Reproducibility
rng = np.random.default_rng(0)

# Settings
d = 100
n_samples = 20000
deltas = [0.5, 2.0, 5.0]
alphas = [2.0, 5.0, 10.0]   # 2.0, 2.5, ..., 10.0

synthetic_data = {}
exact_rows = []

for delta in deltas:
    # Mean vectors
    mu_p = np.zeros(d, dtype=np.float32)
    mu_q = np.zeros(d, dtype=np.float32)
    mu_q[0] = delta   # delta * e1

    # Samples
    x_p = rng.standard_normal(size=(n_samples, d), dtype=np.float32) + mu_p
    x_q = rng.standard_normal(size=(n_samples, d), dtype=np.float32) + mu_q

    synthetic_data[delta] = {
        "P_samples": x_p,   # samples from N(0, I_d)
        "Q_samples": x_q,   # samples from N(delta e1, I_d)
        "mu_p": mu_p,
        "mu_q": mu_q,
    }

    # Exact Renyi divergence values
    for alpha in alphas:
        exact_rows.append({
            "delta": delta,
            "alpha": float(alpha),
            "exact_renyi": float(alpha * delta**2 / 2.0)
        })

exact_df = pd.DataFrame(exact_rows)

# Preview
print("Available deltas:", list(synthetic_data.keys()))
for delta in deltas:
    print(f"delta={delta}: P_samples shape={synthetic_data[delta]['P_samples'].shape}, "
          f"Q_samples shape={synthetic_data[delta]['Q_samples'].shape}")

display(exact_df)

Available deltas: [0.5, 2.0, 5.0]
delta=0.5: P_samples shape=(20000, 100), Q_samples shape=(20000, 100)
delta=2.0: P_samples shape=(20000, 100), Q_samples shape=(20000, 100)
delta=5.0: P_samples shape=(20000, 100), Q_samples shape=(20000, 100)


,delta,alpha,exact_renyi
0,0.5,2.0,0.250
1,0.5,5.0,0.625
2,0.5,10.0,1.250
3,2.0,2.0,4.000
4,2.0,5.0,10.000
5,2.0,10.0,20.000
6,5.0,2.0,25.000
7,5.0,5.0,62.500
8,5.0,10.0,125.000


In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_gpus = 1 if device=='cuda' else 0
print(device)

cpu


In [3]:
class LoadDataset(torch.utils.data.Dataset):
    def __init__(self, N, x, y, x_dim, z_dim):
        self.x = x
        self.y = y
        # file imports and object initialization
        
        self.N = N
        self.x_dim = x_dim
        self.z_dim = z_dim

    def __getitem__(self, ix):
        a, b = self.x[ix,0:self.x_dim], self.y[ix,0:self.z_dim]
        
        return a, b
    
    def __len__(self):
        return self.N

In [ ]:
# Q1/Q2 Gaussian sanity check data
from renyi import Renyi
import torch.nn as nn

dim = 100
alphas = np.arange(2.0, 11.0, 1.0)
deltas = [0.5, 2.0, 5.0]
train_size = 40000
test_size = 10000
seed = 0

def exact_renyi(alpha, delta):
    return 0.5 * alpha * delta**2

gaussian_data = {}
rows = []

for i, delta in enumerate(deltas):
    g = torch.Generator().manual_seed(seed + i)
    q = torch.randn(train_size + test_size, dim, generator=g)
    p = torch.randn(train_size + test_size, dim, generator=g)
    p[:, 0] += delta

    q_train, q_test = q[:train_size], q[train_size:]
    p_train, p_test = p[:train_size], p[train_size:]
    gaussian_data[delta] = (q_train, p_train, q_test, p_test)

    for alpha in alphas:
        rows.append({
            "delta": delta,
            "alpha": alpha,
            "exact": exact_renyi(alpha, delta),
        })

exact_df = pd.DataFrame(rows)
exact_df.pivot(index="alpha", columns="delta", values="exact")


ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
# training loop
batch_size = 5000
epochs = 40
lr = 5e-3
ema_rate = 0.25

class LinearCritic(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Linear(dim, 1)

    def forward(self, x):
        return self.net(x)

results = []

for delta in deltas:
    q_train, p_train, q_test, p_test = gaussian_data[delta]

    train_loader = torch.utils.data.DataLoader(
        LoadDataset(len(q_train), q_train, p_train, x_dim=dim, z_dim=dim),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    for alpha in alphas:
        torch.manual_seed(seed)
        model = Renyi(LinearCritic(dim), renyi_order=alpha, ema_rate=ema_rate).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)

        best_estimate = -float("inf")
        for epoch in range(epochs):
            model.train()
            for q_batch, p_batch in train_loader:
                q_batch = q_batch.to(device)
                p_batch = p_batch.to(device)
                opt.zero_grad()
                loss = model(q_batch, p_batch, update_ema=True)
                loss.backward()
                opt.step()

            with torch.no_grad():
                estimate = float(model.estimate(q_test, p_test).cpu())
            best_estimate = max(best_estimate, estimate)

        exact = exact_renyi(alpha, delta)
        results.append({
            "delta": delta,
            "alpha": alpha,
            "estimate": best_estimate,
            "exact": exact,
        })
        print(f"delta={delta}, alpha={alpha}: estimate={best_estimate:.4f}, exact={exact:.4f}")

results_df = pd.DataFrame(results)
fig, axes = plt.subplots(1, len(deltas), figsize=(15, 4))
for ax, delta in zip(np.atleast_1d(axes), deltas):
    sub = results_df[results_df["delta"] == delta]
    ax.plot(sub["alpha"], sub["estimate"], marker="o", label="estimate")
    ax.plot(sub["alpha"], sub["exact"], linestyle="--", label="exact")
    ax.set_title(f"delta={delta}")
    ax.set_xlabel("alpha")
    ax.grid(alpha=0.3)
axes = np.atleast_1d(axes)
axes[0].set_ylabel("Renyi divergence")
axes[0].legend()
plt.tight_layout()
plt.show()

results_df
